In [ ]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI
import json

documents = load_faq_data()
index = build_index(documents)
llm_client = OpenAI()

def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

response = llm_client.responses.create(
        model = "gpt-5.4-mini",
        input = messages,
        tools = [search_tool]
    )

print(response.output[0])

call = response.output[0]

args = json.loads(call.arguments)

results = search(**args)
json_text = json.dumps(results, indent=2)

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output":json_text
})

response = llm_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text


result = calculate_gpt54mini_price(response.usage.input_tokens, response.usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))



ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join late enrollment enrollment eligibility new student"}', call_id='call_rHqxRXwG6c60YF1lcON0ITTX', name='search', type='function_call', id='fc_02fd4d762983af1e006af77db5736c87d19bbbd96adb8ffc0c', async_=None, caller=None, namespace=None, status='completed')


'Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while the course is still accepting submissions.'